# 06_v2 rolling evaluation
## Fast verification from archived predictions + optional full retraining

This notebook is designed for the HPAI manuscript rolling fiscal-year evaluation.

### Recommended use
1. Run the **FAST VERIFICATION** section first.  
   It does **not** refit the 12 Random Forest models. Instead, it reads the archived prediction parquet files generated by the original analysis and independently recomputes the event-level and pooled metrics.
2. Confirm that the values exactly match the archived summary and the manuscript Table 2 values.
3. Only if necessary, set `RUN_FULL_RETRAIN = True` in the final section. Full retraining fits **12 Random Forest models** on approximately 0.74–1.32 million grid-week rows per fiscal-year split and can take a long time on Colab CPU.

The fast verification is the appropriate first step for checking the integrity of the archived rolling results.

## 1. Mount Google Drive and import packages

In [ ]:
from pathlib import Path
import json
import sys
import time
import platform

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Google Drive mount was skipped:", repr(e))

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Platform:", platform.platform())

## 2. Paths and analysis configuration

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/avian_influenza_project")
PROC_DIR = PROJECT_DIR / "processed"
RESULT_DIR = PROC_DIR / "model_outputs_riskmap_eval"

PANEL_PATH = PROC_DIR / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet"

ARCHIVED_POOLED = RESULT_DIR / "06_v2_rolling_pooled_summary_previous_week_weather.csv"
ARCHIVED_BY_SPLIT = RESULT_DIR / "06_v2_rolling_summary_by_split_previous_week_weather.csv"
ARCHIVED_EVENTS = RESULT_DIR / "06_v2_rolling_event_cases_previous_week_weather.csv"
ARCHIVED_SPLITS = RESULT_DIR / "06_v2_rolling_split_check_previous_week_weather.csv"
ARCHIVED_FEATURES = RESULT_DIR / "06_v2_rolling_feature_sets_previous_week_weather.csv"

MODEL_ORDER = [
    "geo_season_previous_week_weather",
    "geo_season_previous_week_weather_same_grid_lags",
    "geo_season_previous_week_weather_neighbor_past",
    "geo_season_previous_week_weather_lags_neighbor_past",
]

MODEL_LABELS = {
    "geo_season_previous_week_weather":
        "Baseline: geography–seasonality–previous-week temperature",
    "geo_season_previous_week_weather_same_grid_lags":
        "Baseline + same-grid outbreak-history lags",
    "geo_season_previous_week_weather_neighbor_past":
        "Baseline + neighboring outbreak history",
    "geo_season_previous_week_weather_lags_neighbor_past":
        "Baseline + same-grid and neighboring outbreak history",
}

SPLITS = [
    ("test_fy2023", pd.Timestamp("2023-04-01"), pd.Timestamp("2024-04-01")),
    ("test_fy2024", pd.Timestamp("2024-04-01"), pd.Timestamp("2025-04-01")),
    ("test_fy2025", pd.Timestamp("2025-04-01"), pd.Timestamp("2026-04-01")),
]

EXPECTED_GRIDS = 5_491
EXPECTED_EVENTS_PER_MODEL = 61

TARGET_CANDIDATES = [
    "y",
    "target",
    "outbreak",
    "outbreak_flag",
    "is_outbreak",
    "has_outbreak",
    "outbreak_positive",
]

print("RESULT_DIR:", RESULT_DIR)
print("PANEL_PATH:", PANEL_PATH)

## 3. Check that the archived files exist

In [ ]:
required_archived = [
    ARCHIVED_POOLED,
    ARCHIVED_EVENTS,
    ARCHIVED_SPLITS,
    ARCHIVED_FEATURES,
]

missing = [p for p in required_archived if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Required archived files are missing:\n" +
        "\n".join(str(p) for p in missing)
    )

prediction_paths = {}
for split_name, _, _ in SPLITS:
    for model_name in MODEL_ORDER:
        path = RESULT_DIR / f"06_v2_rolling_{split_name}_{model_name}_predictions.parquet"
        prediction_paths[(split_name, model_name)] = path

missing_preds = [p for p in prediction_paths.values() if not p.exists()]

if missing_preds:
    raise FileNotFoundError(
        "Archived prediction parquet files are missing:\n" +
        "\n".join(str(p) for p in missing_preds)
    )

print("All required archived files were found.")
print("Prediction parquet files:", len(prediction_paths))

## 4. Load the archived reference summaries

In [ ]:
archived_pooled = pd.read_csv(ARCHIVED_POOLED)
archived_by_split = (
    pd.read_csv(ARCHIVED_BY_SPLIT)
    if ARCHIVED_BY_SPLIT.exists()
    else None
)
archived_events = pd.read_csv(ARCHIVED_EVENTS)
archived_splits = pd.read_csv(ARCHIVED_SPLITS)
archived_features = pd.read_csv(ARCHIVED_FEATURES)

print("Archived pooled summary")
display(archived_pooled)

print("\nArchived fiscal-year split definition")
display(archived_splits)

print("\nArchived feature sets")
display(archived_features)

# FAST VERIFICATION

The next cells read the 12 archived prediction parquet files produced by the original analysis. No model is refitted.

## 5. Helper functions

In [ ]:
def find_target_column(df):
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            values = set(pd.Series(df[c]).dropna().unique().tolist())
            if values.issubset({0, 1, False, True}):
                return c

    if "outbreak_count" in df.columns:
        return "__outbreak_count__"

    candidates = []
    for c in df.columns:
        try:
            values = set(pd.Series(df[c]).dropna().unique().tolist())
            if len(values) <= 2 and values.issubset({0, 1, False, True}):
                candidates.append(c)
        except Exception:
            pass

    raise ValueError(
        "Could not identify the outbreak target column.\n"
        f"Columns: {df.columns.tolist()}\n"
        f"Binary-like candidates: {candidates}"
    )


def ensure_weekly_rank_metrics(df):
    df = df.copy()

    if "week_start" not in df.columns:
        raise KeyError("week_start is missing.")

    df["week_start"] = pd.to_datetime(df["week_start"])

    if "risk_rank" in df.columns and "risk_percentile" in df.columns:
        return df

    if "pred_proba" not in df.columns:
        raise KeyError(
            "Neither archived risk_rank/risk_percentile nor pred_proba was found."
        )

    parts = []

    for week, g0 in df.groupby("week_start", sort=True):
        g = g0.copy()
        n = len(g)

        if n <= 1:
            raise ValueError(f"Invalid grid count in week {week}: {n}")

        g["n_grids_in_week"] = n
        g["risk_rank"] = g["pred_proba"].rank(
            method="average",
            ascending=False,
        )
        g["risk_percentile"] = (
            1.0 - (g["risk_rank"] - 1.0) / (n - 1.0)
        )

        parts.append(g)

    return pd.concat(parts, ignore_index=True)


def top_hit_series(df, p):
    candidates = [
        f"top{p}",
        f"top{p}_hit",
        f"top_{p}",
        f"top_{p}_hit",
    ]

    for c in candidates:
        if c in df.columns:
            return df[c].astype(bool)

    cutoff = int(np.ceil(EXPECTED_GRIDS * (p / 100.0)))
    return df["risk_rank"] <= cutoff


def summarize_event_rows(event_df):
    rows = []

    for model_name in MODEL_ORDER:
        g = event_df.loc[
            event_df["model_name"].eq(model_name)
        ].copy()

        row = {
            "model_name": model_name,
            "display_label": MODEL_LABELS[model_name],
            "n_splits": int(g["split_name"].nunique()),
            "total_events": int(len(g)),
            "mean_event_percentile": float(g["risk_percentile"].mean()),
            "median_event_percentile": float(g["risk_percentile"].median()),
            "mean_event_rank": float(g["risk_rank"].mean()),
            "median_event_rank": float(g["risk_rank"].median()),
        }

        for p in [1, 5, 10, 20]:
            hits = top_hit_series(g, p)
            row[f"top{p}_events"] = int(hits.sum())
            row[f"top{p}_capture_rate"] = float(hits.mean())

        rows.append(row)

    return pd.DataFrame(rows)

## 6. Read all 12 archived prediction files

In [ ]:
event_parts = []
inventory_rows = []

for split_name, _, _ in SPLITS:
    for model_name in MODEL_ORDER:

        path = prediction_paths[(split_name, model_name)]

        t0 = time.time()
        df = pd.read_parquet(path)
        df = ensure_weekly_rank_metrics(df)

        target_col = find_target_column(df)

        if target_col == "__outbreak_count__":
            positive_mask = pd.to_numeric(
                df["outbreak_count"],
                errors="coerce",
            ).fillna(0).gt(0)
        else:
            positive_mask = pd.to_numeric(
                df[target_col],
                errors="coerce",
            ).fillna(0).astype(int).eq(1)

        ev = df.loc[positive_mask].copy()
        ev["split_name"] = split_name
        ev["model_name"] = model_name
        ev["display_label"] = MODEL_LABELS[model_name]

        event_parts.append(ev)

        grid_counts = df.groupby("week_start").size()

        inventory_rows.append({
            "split_name": split_name,
            "model_name": model_name,
            "rows": len(df),
            "weeks": int(df["week_start"].nunique()),
            "event_rows": int(len(ev)),
            "grids_per_week_min": int(grid_counts.min()),
            "grids_per_week_max": int(grid_counts.max()),
            "seconds_to_read": time.time() - t0,
            "target_column": target_col,
        })

        print(
            f"{split_name:11s} | "
            f"{model_name:55s} | "
            f"rows={len(df):,} | "
            f"events={len(ev):2d} | "
            f"{time.time()-t0:.1f}s"
        )

inventory = pd.DataFrame(inventory_rows)
event_cases_verified = pd.concat(
    event_parts,
    ignore_index=True,
)

print("\nPrediction inventory")
display(inventory)

## 7. Structural validation of the archived predictions

In [ ]:
errors = []

if len(inventory) != 12:
    errors.append(
        f"Expected 12 split-model combinations, found {len(inventory)}."
    )

if not inventory["event_rows"].groupby(
    inventory["model_name"]
).sum().eq(EXPECTED_EVENTS_PER_MODEL).all():
    observed = (
        inventory.groupby("model_name")["event_rows"]
        .sum()
        .to_dict()
    )
    errors.append(
        f"Expected 61 pooled events per model; observed {observed}"
    )

if not (
    inventory["grids_per_week_min"].eq(EXPECTED_GRIDS).all()
    and inventory["grids_per_week_max"].eq(EXPECTED_GRIDS).all()
):
    errors.append(
        "At least one archived prediction file does not contain "
        "exactly 5,491 grids in every evaluation week."
    )

if errors:
    raise AssertionError("\n".join(errors))

print("PASS: 12 split-model combinations")
print("PASS: 61 pooled outbreak events for each model")
print("PASS: 5,491 grid cells in every evaluation week")

## 8. Recompute the pooled Table 2 metrics from the archived predictions

In [ ]:
verified_pooled = summarize_event_rows(
    event_cases_verified
)

display_cols = [
    "model_name",
    "total_events",
    "top1_events",
    "top1_capture_rate",
    "top5_events",
    "top5_capture_rate",
    "top10_events",
    "top10_capture_rate",
    "top20_events",
    "top20_capture_rate",
    "mean_event_percentile",
    "median_event_percentile",
    "mean_event_rank",
    "median_event_rank",
]

display(verified_pooled[display_cols])

## 9. Exact comparison with the archived pooled summary

In [ ]:
metric_columns = [
    "total_events",
    "mean_event_percentile",
    "median_event_percentile",
    "mean_event_rank",
    "median_event_rank",
    "top1_events",
    "top1_capture_rate",
    "top5_events",
    "top5_capture_rate",
    "top10_events",
    "top10_capture_rate",
    "top20_events",
    "top20_capture_rate",
]

ref = archived_pooled[
    ["model_name"] + metric_columns
].copy()

obs = verified_pooled[
    ["model_name"] + metric_columns
].copy()

cmp = obs.merge(
    ref,
    on="model_name",
    suffixes=("_verified", "_archived"),
    validate="one_to_one",
)

comparison_rows = []

for _, r in cmp.iterrows():
    model_name = r["model_name"]

    for metric in metric_columns:
        a = r[f"{metric}_verified"]
        b = r[f"{metric}_archived"]

        if metric.endswith("_events") or metric == "total_events":
            passed = int(a) == int(b)
            delta = int(a) - int(b)
        else:
            passed = bool(
                np.isclose(
                    float(a),
                    float(b),
                    atol=1e-12,
                    rtol=0,
                )
            )
            delta = float(a) - float(b)

        comparison_rows.append({
            "model_name": model_name,
            "metric": metric,
            "verified": a,
            "archived": b,
            "delta": delta,
            "passed": passed,
        })

comparison = pd.DataFrame(comparison_rows)

failed = comparison.loc[~comparison["passed"]]

if len(failed):
    display(failed)
    raise AssertionError(
        "Fast verification did not exactly match "
        "the archived pooled summary."
    )

print(
    "PASS: all pooled metrics recomputed from the archived "
    "prediction parquet files exactly match the archived summary."
)

## 10. Direct validation against the manuscript Table 2 values

In [ ]:
MANUSCRIPT_EXPECTED = {
    "geo_season_previous_week_weather": {
        "top1": 0.04918032786885246,
        "top5": 0.16393442622950818,
        "top10": 0.2786885245901639,
        "top20": 0.45901639344262296,
        "mean_percentile": 0.7132774644649518,
        "median_percentile": 0.7861955927881988,
    },
    "geo_season_previous_week_weather_same_grid_lags": {
        "top1": 0.04918032786885246,
        "top5": 0.13114754098360656,
        "top10": 0.26229508196721313,
        "top20": 0.45901639344262296,
        "mean_percentile": 0.7123758400482458,
        "median_percentile": 0.7787288289928975,
    },
    "geo_season_previous_week_weather_neighbor_past": {
        "top1": 0.03278688524590164,
        "top5": 0.18032786885245902,
        "top10": 0.26229508196721313,
        "top20": 0.45901639344262296,
        "mean_percentile": 0.7023534785685069,
        "median_percentile": 0.7894736842105263,
    },
    "geo_season_previous_week_weather_lags_neighbor_past": {
        "top1": 0.06557377049180328,
        "top5": 0.14754098360655737,
        "top10": 0.21311475409836064,
        "top20": 0.4426229508196721,
        "mean_percentile": 0.6950419613615125,
        "median_percentile": 0.7326534328901839,
    },
}

validation_rows = []

for model_name, exp in MANUSCRIPT_EXPECTED.items():

    row = verified_pooled.loc[
        verified_pooled["model_name"].eq(model_name)
    ].iloc[0]

    observed = {
        "top1": float(row["top1_capture_rate"]),
        "top5": float(row["top5_capture_rate"]),
        "top10": float(row["top10_capture_rate"]),
        "top20": float(row["top20_capture_rate"]),
        "mean_percentile": float(row["mean_event_percentile"]),
        "median_percentile": float(row["median_event_percentile"]),
    }

    for metric, expected_value in exp.items():
        observed_value = observed[metric]

        validation_rows.append({
            "model_name": model_name,
            "metric": metric,
            "observed": observed_value,
            "expected": expected_value,
            "passed": bool(
                np.isclose(
                    observed_value,
                    expected_value,
                    atol=1e-12,
                    rtol=0,
                )
            ),
        })

manuscript_validation = pd.DataFrame(
    validation_rows
)

display(manuscript_validation)

if not manuscript_validation["passed"].all():
    raise AssertionError(
        "At least one verified value differs from "
        "the manuscript reference value."
    )

print("\n" + "=" * 72)
print("SUCCESS")
print("ARCHIVED 06_v2 PREDICTIONS REPRODUCE THE MANUSCRIPT TABLE 2 VALUES.")
print("=" * 72)

## 11. Manuscript-friendly summary

In [ ]:
publication_table = verified_pooled.copy()

publication_table["Top 1%"] = (
    100 * publication_table["top1_capture_rate"]
).map(lambda x: f"{x:.1f}%")

publication_table["Top 5%"] = (
    100 * publication_table["top5_capture_rate"]
).map(lambda x: f"{x:.1f}%")

publication_table["Top 10%"] = (
    100 * publication_table["top10_capture_rate"]
).map(lambda x: f"{x:.1f}%")

publication_table["Top 20%"] = (
    100 * publication_table["top20_capture_rate"]
).map(lambda x: f"{x:.1f}%")

publication_table["Mean percentile"] = publication_table[
    "mean_event_percentile"
].map(lambda x: f"{x:.3f}")

publication_table["Median percentile"] = publication_table[
    "median_event_percentile"
].map(lambda x: f"{x:.3f}")

publication_table["Mean rank"] = publication_table[
    "mean_event_rank"
].map(lambda x: f"{x:,.1f}")

publication_table["Median rank"] = publication_table[
    "median_event_rank"
].map(lambda x: f"{x:,.1f}")

publication_table = publication_table[
    [
        "display_label",
        "total_events",
        "Top 1%",
        "Top 5%",
        "Top 10%",
        "Top 20%",
        "Mean percentile",
        "Median percentile",
        "Mean rank",
        "Median rank",
    ]
].rename(
    columns={
        "display_label": "Predictor set",
        "total_events": "Events",
    }
)

display(publication_table)

## 12. Save verification outputs

In [ ]:
VERIFY_DIR = RESULT_DIR / "06_v2_public_reproducibility_verification"
VERIFY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

inventory.to_csv(
    VERIFY_DIR / "prediction_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

verified_pooled.to_csv(
    VERIFY_DIR / "verified_pooled_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    VERIFY_DIR / "verified_vs_archived_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

manuscript_validation.to_csv(
    VERIFY_DIR / "verified_vs_manuscript_table2.csv",
    index=False,
    encoding="utf-8-sig",
)

publication_table.to_csv(
    VERIFY_DIR / "manuscript_friendly_table2.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved verification outputs to:")
print(VERIFY_DIR)

# OPTIONAL: full retraining

**Do not run this section unless full refitting is specifically required.**

The code below retrains all 12 Random Forest models using the model-ready panel. It is intentionally disabled by default because it can require a long CPU run.

Important:
- The fast verification above checks the actual archived predictions used for the reported rolling evaluation.
- Full refitting can be sensitive to the exact software environment, row order, and implementation details.
- If the objective is to document the reported results, preserve the archived prediction-based verification as the primary validation record.

## 13. Full-retraining configuration

In [ ]:
RUN_FULL_RETRAIN = False

RF_N_ESTIMATORS = 300
RF_MAX_DEPTH = 12
RF_MIN_SAMPLES_LEAF = 10
RF_CLASS_WEIGHT = "balanced"
RF_RANDOM_STATE = 42

print("RUN_FULL_RETRAIN =", RUN_FULL_RETRAIN)

## 14. Full retraining helper

In [ ]:
if RUN_FULL_RETRAIN:

    import joblib

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
    )

    if not PANEL_PATH.exists():
        raise FileNotFoundError(PANEL_PATH)

    panel = pd.read_parquet(PANEL_PATH)
    panel["week_start"] = pd.to_datetime(
        panel["week_start"]
    )

    if "y" in panel.columns:
        TARGET = "y"
    elif "target" in panel.columns:
        TARGET = "target"
    elif "outbreak" in panel.columns:
        TARGET = "outbreak"
    elif "outbreak_flag" in panel.columns:
        TARGET = "outbreak_flag"
    elif "outbreak_count" in panel.columns:
        panel["_target_from_outbreak_count"] = (
            pd.to_numeric(
                panel["outbreak_count"],
                errors="coerce",
            )
            .fillna(0)
            .gt(0)
            .astype(int)
        )
        TARGET = "_target_from_outbreak_count"
    else:
        raise KeyError(
            "Could not identify the target in the model-ready panel."
        )

    feature_table = pd.read_csv(
        ARCHIVED_FEATURES
    )

    MODEL_SPECS = {}

    for _, r in feature_table.iterrows():
        feats = [
            x.strip()
            for x in str(r["features"]).split(";")
            if x.strip()
        ]

        MODEL_SPECS[r["model_name"]] = (
            r["display_label"],
            feats,
        )

    print("Target:", TARGET)
    print("Panel rows:", f"{len(panel):,}")
    print("Panel events:", int(panel[TARGET].sum()))

    for model_name in MODEL_ORDER:
        label, feats = MODEL_SPECS[model_name]
        missing = [
            c for c in feats
            if c not in panel.columns
        ]
        if missing:
            raise KeyError(
                f"{model_name}: missing features {missing}"
            )

    print("Feature validation passed.")
else:
    print(
        "Skipped. Set RUN_FULL_RETRAIN = True "
        "to enable full retraining."
    )

## 15. Full retraining of the 12 Random Forest models

In [ ]:
if RUN_FULL_RETRAIN:

    RETRAIN_DIR = (
        RESULT_DIR
        / "06_v2_full_retraining_check"
    )
    RETRAIN_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    retrain_summary_rows = []
    retrain_event_parts = []
    retrain_feature_importance = []

    for split_name, test_start, test_end in SPLITS:

        train = panel.loc[
            panel["week_start"] < test_start
        ].copy()

        test = panel.loc[
            (panel["week_start"] >= test_start)
            & (panel["week_start"] < test_end)
        ].copy()

        print("\n" + "=" * 72)
        print(
            split_name,
            "train rows =", f"{len(train):,}",
            "train events =", int(train[TARGET].sum()),
            "test rows =", f"{len(test):,}",
            "test events =", int(test[TARGET].sum()),
        )
        print("=" * 72)

        for model_name in MODEL_ORDER:

            display_label, features = MODEL_SPECS[
                model_name
            ]

            med = train[features].median(
                numeric_only=True
            )

            X_train = train[features].fillna(med)
            X_test = test[features].fillna(med)

            y_train = train[TARGET].astype(int)
            y_test = test[TARGET].astype(int)

            rf = RandomForestClassifier(
                n_estimators=RF_N_ESTIMATORS,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
                class_weight=RF_CLASS_WEIGHT,
                random_state=RF_RANDOM_STATE,
                n_jobs=-1,
                verbose=0,
            )

            print(
                f"\nSTART fitting: "
                f"{split_name} / {model_name}"
            )
            print(
                f"Training rows: {len(X_train):,}; "
                f"features: {len(features)}"
            )

            t0 = time.time()

            rf.fit(
                X_train,
                y_train,
            )

            fit_minutes = (
                time.time() - t0
            ) / 60.0

            print(
                f"FINISHED fitting: "
                f"{split_name} / {model_name} "
                f"in {fit_minutes:.1f} min"
            )

            score = rf.predict_proba(
                X_test
            )[:, 1]

            keep = [
                "grid_id",
                "week_start",
                TARGET,
                "grid_lat",
                "grid_lon",
            ]

            if "outbreak_count" in test.columns:
                keep.append("outbreak_count")

            pred = test[keep].copy()
            pred["pred_proba"] = score
            pred["split_name"] = split_name
            pred["model_name"] = model_name
            pred["display_label"] = display_label

            pred = ensure_weekly_rank_metrics(
                pred
            )

            for p in [1, 5, 10, 20]:
                cutoff = int(
                    np.ceil(
                        EXPECTED_GRIDS * (p / 100.0)
                    )
                )
                pred[f"top{p}"] = (
                    pred["risk_rank"] <= cutoff
                )

            pred_path = (
                RETRAIN_DIR
                / f"06_v2_retrained_{split_name}_"
                  f"{model_name}_predictions.parquet"
            )

            model_path = (
                RETRAIN_DIR
                / f"06_v2_retrained_{split_name}_"
                  f"{model_name}.joblib"
            )

            pred.to_parquet(
                pred_path,
                index=False,
            )

            joblib.dump(
                rf,
                model_path,
            )

            ev = pred.loc[
                pred[TARGET].eq(1)
            ].copy()

            retrain_event_parts.append(ev)

            base = float(y_test.mean())

            row = {
                "split_name": split_name,
                "model_name": model_name,
                "display_label": display_label,
                "test_start": test_start.date().isoformat(),
                "test_end_exclusive": test_end.date().isoformat(),
                "test_rows": len(test),
                "n_events": int(y_test.sum()),
                "baseline_event_rate": base,
                "roc_auc": roc_auc_score(
                    y_test,
                    score,
                ),
                "pr_auc": average_precision_score(
                    y_test,
                    score,
                ),
                "mean_event_percentile":
                    ev["risk_percentile"].mean(),
                "median_event_percentile":
                    ev["risk_percentile"].median(),
                "mean_event_rank":
                    ev["risk_rank"].mean(),
                "median_event_rank":
                    ev["risk_rank"].median(),
                "pr_auc_over_baseline":
                    average_precision_score(
                        y_test,
                        score,
                    ) / base,
                "n_features": len(features),
                "fit_minutes": fit_minutes,
            }

            for p in [1, 5, 10, 20]:
                row[f"top{p}_events"] = int(
                    ev[f"top{p}"].sum()
                )
                row[f"top{p}_capture_rate"] = float(
                    ev[f"top{p}"].mean()
                )

            retrain_summary_rows.append(row)

            for f, imp in zip(
                features,
                rf.feature_importances_,
            ):
                retrain_feature_importance.append({
                    "split_name": split_name,
                    "model_name": model_name,
                    "display_label": display_label,
                    "feature": f,
                    "importance": float(imp),
                })

    retrain_summary_by_split = pd.DataFrame(
        retrain_summary_rows
    )

    retrain_event_cases = pd.concat(
        retrain_event_parts,
        ignore_index=True,
    )

    retrain_feature_importance = pd.DataFrame(
        retrain_feature_importance
    )

    retrain_summary_by_split.to_csv(
        RETRAIN_DIR
        / "06_v2_retrained_summary_by_split.csv",
        index=False,
        encoding="utf-8-sig",
    )

    retrain_event_cases.to_csv(
        RETRAIN_DIR
        / "06_v2_retrained_event_cases.csv",
        index=False,
        encoding="utf-8-sig",
    )

    retrain_feature_importance.to_csv(
        RETRAIN_DIR
        / "06_v2_retrained_feature_importance.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("\nFull retraining completed.")
    print("Saved to:", RETRAIN_DIR)

else:
    print(
        "Full retraining was not run. "
        "This is expected in normal verification mode."
    )

## 16. Optional comparison of full-retraining results with the archived results

In [ ]:
if RUN_FULL_RETRAIN:

    pooled_rows = []

    for model_name in MODEL_ORDER:
        g = retrain_event_cases.loc[
            retrain_event_cases["model_name"].eq(
                model_name
            )
        ].copy()

        row = {
            "model_name": model_name,
            "total_events": len(g),
            "mean_event_percentile":
                g["risk_percentile"].mean(),
            "median_event_percentile":
                g["risk_percentile"].median(),
            "mean_event_rank":
                g["risk_rank"].mean(),
            "median_event_rank":
                g["risk_rank"].median(),
        }

        for p in [1, 5, 10, 20]:
            row[f"top{p}_events"] = int(
                g[f"top{p}"].sum()
            )
            row[f"top{p}_capture_rate"] = float(
                g[f"top{p}"].mean()
            )

        pooled_rows.append(row)

    retrained_pooled = pd.DataFrame(
        pooled_rows
    )

    compare = retrained_pooled.merge(
        archived_pooled,
        on="model_name",
        suffixes=("_retrained", "_archived"),
        validate="one_to_one",
    )

    cols = [
        "model_name",
        "top10_capture_rate_retrained",
        "top10_capture_rate_archived",
        "top20_capture_rate_retrained",
        "top20_capture_rate_archived",
        "mean_event_percentile_retrained",
        "mean_event_percentile_archived",
    ]

    display(compare[cols])

    print(
        "\nNOTE: if the refitted values differ, "
        "do not overwrite the archived reported outputs. "
        "Investigate software environment, row order, "
        "and exact original training provenance first."
    )

else:
    print("Skipped because RUN_FULL_RETRAIN = False.")

## Interpretation of the verification result

If the notebook prints:

`SUCCESS — ARCHIVED 06_v2 PREDICTIONS REPRODUCE THE MANUSCRIPT TABLE 2 VALUES.`

then the archived prediction files independently reproduce the rolling-performance values reported in the manuscript.

For public code release, this notebook can be used as the **fast validation entry point**. The optional full-retraining section should be described separately because it is computationally intensive and may depend on the exact original software/runtime and row-order provenance.